# BA283 GRU training on Colab T4

Runtime -> Change runtime type -> **T4 GPU**, then run cells top to bottom.

1. Upload `ba283_train_bundle.zip` (made locally with `colab/make_bundle.sh`).
2. Stages the bundle to local `/content/dataset` and remaps manifests to `/content`, then trains GRU(2,32)+Dense(1), 200 epochs, patience 30.
3. Exports `ba283_gru32.json` (self-describing weights) + downloads it and `best.pth`.

Drop the downloaded `ba283_gru32.json` into the repo at `plugin/weights/ba283_gru32.json` to replace the placeholder.

In [ ]:
from google.colab import files
files.upload()  # pick ba283_train_bundle.zip

In [ ]:
!unzip -q -o ba283_train_bundle.zip -d /content/ba283
%cd /content/ba283
!pip -q install soundfile pandas
from train.stage import stage_zip, remap_manifests
DATA = str(stage_zip('/content/ba283_train_bundle.zip', '/content/dataset'))
M = remap_manifests(DATA, out_dir='/content')
print('staged ->', DATA)
print(M)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime type to T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
!python -m train.train \
  --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 5e-3 --hidden 32 --device cuda \
  --checkpoint-dir ckpt --log-dir logs --patience 999 --seed 42

In [ ]:
from pathlib import Path
import torch
from train.model import BA283LSTM
from train.export import export_rtneural_json

m = BA283LSTM(hidden=32)
m.load_state_dict(torch.load('ckpt/best.pth', map_location='cpu', weights_only=True))
m.eval()
meta = {
    'source': 'colab-t4',
    'epochs': 200,
    'dataset': '45 sims (9 stim x 5 bias), multisine dropped',
    'hidden': 32,
    'gate_order': 'rzn',
}
export_rtneural_json(m, Path('ba283_gru32.json'), meta)
print('exported', Path('ba283_gru32.json').stat().st_size, 'bytes')

In [ ]:
from google.colab import files
files.download('ba283_gru32.json')
files.download('ckpt/best.pth')
files.download('ckpt/last.pth')
